In [3]:
import math
import tkinter as tk
from tkinter import messagebox


def tvm(n=None, r=None, pv=None, pmt=None, fv=None):
    """
    Solve for whichever TVM variable is None.
    Equation (end-of-period payments, calculator sign convention):
        PV*(1+r)^n + PMT*((1+r)^n - 1)/r + FV = 0
    Outflows are negative, inflows positive.
    """
    args = {"n": n, "r": r, "pv": pv, "pmt": pmt, "fv": fv}
    missing = [k for k, v in args.items() if v is None]
    if len(missing) != 1:
        raise ValueError(f"Leave exactly ONE box blank. Blank now: {missing}")
    target = missing[0]

    if target != "r" and r == 0:
        raise ValueError("Rate of 0 is not supported.")

    if target == "fv":
        factor = (1 + r) ** n
        return -(pv * factor + pmt * (factor - 1) / r)

    if target == "pv":
        factor = (1 + r) ** n
        return -(fv + pmt * (factor - 1) / r) / factor

    if target == "pmt":
        factor = (1 + r) ** n
        return -(pv * factor + fv) / ((factor - 1) / r)

    if target == "n":
        growth = (pmt / r - fv) / (pv + pmt / r)
        if growth <= 0:
            raise ValueError("No solution for N with these inputs (check signs).")
        return math.log(growth) / math.log(1 + r)

    if target == "r":
        def f(rate):
            # The TVM equation itself; equals 0 at the correct rate   <-- was TODO 1
            factor = (1 + rate) ** n
            return pv * factor + pmt * (factor - 1) / rate + fv

        low, high = 1e-9, 1.0
        if f(low) * f(high) > 0:
            raise ValueError("No rate found between 0% and 100% (check signs).")
        for _ in range(200):
            mid = (low + high) / 2
            if f(low) * f(mid) > 0:   # same sign -> root is in upper half   <-- was TODO 2
                low = mid
            else:
                high = mid
        return (low + high) / 2


# ---------- GUI ----------
FIELDS = ["n", "r", "pv", "pmt", "fv"]
LABELS = {"n": "N (periods)", "r": "R (rate per period, e.g. 0.06)",
          "pv": "Present Value", "pmt": "Payment", "fv": "Future Value"}


def read_box(entry):
    text = entry.get().strip()
    return None if text == "" else float(text)


def on_calculate():
    try:
        values = {name: read_box(entries[name]) for name in FIELDS}
        result = tvm(**values)
        missing = [k for k, v in values.items() if v is None][0]
        if missing == "r":
            result_label.config(text=f"R = {result:.4%} per period")
        else:
            result_label.config(text=f"{missing.upper()} = {result:,.4f}")
    except ValueError as e:
        messagebox.showerror("Input error", str(e))
    except ZeroDivisionError:
        messagebox.showerror("Input error", "Division by zero — check your inputs.")


root = tk.Tk()
root.title("TVM Calculator")

entries = {}
for row, name in enumerate(FIELDS):
    tk.Label(root, text=LABELS[name]).grid(row=row, column=0, padx=8, pady=4, sticky="e")
    entry = tk.Entry(root, width=20)
    entry.grid(row=row, column=1, padx=8, pady=4)
    entries[name] = entry

tk.Button(root, text="Calculate", command=on_calculate).grid(
    row=len(FIELDS), column=0, columnspan=2, pady=10
)
result_label = tk.Label(root, text="Leave one box blank, then click Calculate.")
result_label.grid(row=len(FIELDS) + 1, column=0, columnspan=2, pady=5)

root.mainloop()